# PREPROCESSING PIPELINES NOTEBOOK

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, classification_report
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from category_encoders import TargetEncoder

import joblib

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

## 1. Load the dataset

In [ ]:
DATA_PATH = '../data/train.csv'
TARGET_COLUMN = 'Irrigation_Need'
SUBMISSION_PATH = '../data/submission.csv'
SUBMISSION_FILES = Path('../data/submission')
ID_COLUMN = 'id'
TASK_TYPE = 'auto'
TEST_SIZE = 0.2
RANDOM_STATE = 42

def load_data(path):
    return pd.read_csv(path)

df = load_data(DATA_PATH)
display(df.head())

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


## 2. Basic checks

In [3]:
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found in the dataset.")

summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'null_count': df.isnull().sum(),
    'null_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique(dropna=True)
}).sort_values(['null_pct', 'n_unique'], ascending=[False, False])

display(summary)
print('\nTarget preview:')
display(df[TARGET_COLUMN].head())

,dtype,null_count,null_pct,n_unique
id,int64,0,0.0,630000
Rainfall_mm,float64,0,0.0,19308
Previous_Irrigation_mm,float64,0,0.0,10110
Humidity,float64,0,0.0,6475
Soil_Moisture,float64,0,0.0,5223
Temperature_C,float64,0,0.0,2934
Wind_Speed_kmh,float64,0,0.0,1935
Field_Area_hectare,float64,0,0.0,1466
Sunlight_Hours,float64,0,0.0,701
Soil_pH,float64,0,0.0,341



Target preview:


0       Low
1       Low
2       Low
3    Medium
4       Low
Name: Irrigation_Need, dtype: object

## 3. Split features and target

In [4]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print('Feature shape:', X.shape)
print('Target shape:', y.shape)

Feature shape: (630000, 20)
Target shape: (630000,)


## 4. Detect numeric and categorical

In [5]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Numeric features: ['id', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
Categorical features: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


## 5. Build preprocessing pipelines

In [6]:
target_encode_cols = [
    'Crop_Type',
    'Soil_Type'
]

onehot_cols = [col for col in categorical_features if col not in target_encode_cols]

In [19]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

target_encode_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target', TargetEncoder())
])

onehot_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

soil_region_freq_pipeline = Pipeline([
    ('freq')
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('target', target_encode_pipeline, categorical_features),
])

preprocessor

,transformers,"[('num', ...), ('target', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [20]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df[TARGET_COLUMN].astype(str))

label_encoder

LabelEncoder()

## 6. Train/test split

In [21]:
def infer_task_type(y_series):
    if TASK_TYPE in {'regression', 'classification'}:
        return TASK_TYPE
    if pd.api.types.is_numeric_dtype(y_series):
        unique_ratio = y_series.nunique(dropna=True) / max(len(y_series), 1)
        if y_series.nunique(dropna=True) <= 10 or unique_ratio < 0.05:
            return 'classification'
        return 'regression'
    return 'classification'

In [22]:
task_type = infer_task_type(y)
print('Inferred task type:', task_type)

Inferred task type: classification


In [23]:
stratify_y = y if task_type == 'classification' and y.nunique(dropna=True) > 1 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE, 
    stratify=stratify_y
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

X_train: (504000, 20)
X_test: (126000, 20)
y_train: (504000,)
y_test: (126000,)


## 7. Build full training pipeline

In [24]:
if task_type == 'regression':
    model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
else:
    model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

rand_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

rand_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('target', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 8. Fit the pipeline

In [25]:
rand_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('target', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 9. Evaluate on test data

In [26]:
y_pred = rand_pipeline.predict(X_test)

if task_type == 'regression':
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"Regression Metrics:\nMSE: {mse:.4f}\nMAE: {mae:.4f}\nR²: {r2:.4f}")
else:
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    print(f"Classification Metrics:\nAccuracy: {acc:.4f}\n\nClassification Report:\n{report}")

Classification Metrics:
Accuracy: 0.9860

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.92      0.94      4202
           1       0.99      1.00      0.99     73983
           2       0.99      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



## 10. Inspect transformed feature names

In [27]:
feature_names = rand_pipeline.named_steps['preprocessor'].get_feature_names_out()
print(f'Total transformed features: {len(feature_names)}')
display(pd.DataFrame({
    'feature': feature_names,
    'importance': rand_pipeline.named_steps['model'].feature_importances_
}).sort_values('importance', ascending=False).head(20))

Total transformed features: 20


,feature,importance
14,target__2,0.286752
2,num__Soil_Moisture,0.278638
9,num__Wind_Speed_kmh,0.113132
18,target__6,0.107788
5,num__Temperature_C,0.106565
7,num__Rainfall_mm,0.038117
11,num__Previous_Irrigation_mm,0.010679
6,num__Humidity,0.008689
10,num__Field_Area_hectare,0.006509
0,num__id,0.006329


In [29]:
feature_names = rand_pipeline.named_steps['preprocessor'].get_feature_names_out()
print(f'Total transformed features: {len(feature_names)}')
display(pd.DataFrame({'feature_name': feature_names}).head(10))

Total transformed features: 20


,feature_name
0,num__id
1,num__Soil_pH
2,num__Soil_Moisture
3,num__Organic_Carbon
4,num__Electrical_Conductivity
5,num__Temperature_C
6,num__Humidity
7,num__Rainfall_mm
8,num__Sunlight_Hours
9,num__Wind_Speed_kmh


## Experiments

In [41]:
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.pipeline import Pipeline

In [42]:
if task_type == 'regression':
    models = {
        'RandomForest': RandomForestRegressor(
            random_state=RANDOM_STATE, 
            n_jobs=-1
        ),
        'LightGBM': LGBMRegressor(
            random_state=RANDOM_STATE,
            n_estimators=300,
            learning_rate=0.05,
            n_jobs=-1
        ),
        'CatBoost': CatBoostRegressor(
            random_state=RANDOM_STATE,
            iterations=300,
            learning_rate=0.05,
            verbose=0
        ),
        'XGBoost': XGBRegressor(
            random_state=RANDOM_STATE,
            n_estimators=300,
            learning_rate=0.05,
            n_jobs=-1,
            eval_metric='rmse'
        )
    }
else:
    models = {
        'RandomForest': RandomForestClassifier(
            random_state=RANDOM_STATE, 
            n_jobs=-1
        ),
        'LightGBM': LGBMClassifier(
            random_state=RANDOM_STATE,
            n_estimators=1000,
            learning_rate=0.05,
            n_jobs=-1
        ),
        'CatBoost': CatBoostClassifier(
            random_state=RANDOM_STATE,
            iterations=1000,
            learning_rate=0.05,
            verbose=0
        ),
        'XGBoost': XGBClassifier(
            random_state=RANDOM_STATE,
            n_estimators=1000,
            learning_rate=0.05,
            n_jobs=-1,
            eval_metric='mlogloss'
        )
    }

pipelines = {
    name: Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ]) for name, model in models.items()
}

In [43]:
for name, pipe in pipelines.items():
    print(f"Training {name}...")
    pipe.fit(X_train, y_train)

Training RandomForest...
Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003884 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2968
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 20
[LightGBM] [Info] Start training from score -3.400781
[LightGBM] [Info] Start training from score -0.532440
[LightGBM] [Info] Start training from score -0.968948
Training CatBoost...
Training XGBoost...


In [45]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

for name, pipe in pipelines.items():
    y_pred = pipe.predict(X_test)
    print(f"\n=== {name.upper()} ===")
    
    if task_type == 'regression':
        rmse = mean_squared_error(y_test, y_pred, squared=False)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE:  {mae:.4f}")
        print(f"R²:   {r2:.4f}")
    else:
        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy: {acc:.4f}")
        print(classification_report(y_test, y_pred))


=== RANDOMFOREST ===
Accuracy: 0.9860
              precision    recall  f1-score   support

           0       0.97      0.92      0.94      4202
           1       0.99      1.00      0.99     73983
           2       0.99      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000


=== LIGHTGBM ===
Accuracy: 0.9849
              precision    recall  f1-score   support

           0       0.96      0.92      0.94      4202
           1       0.99      0.99      0.99     73983
           2       0.98      0.98      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.98      0.98      0.98    126000


=== CATBOOST ===
Accuracy: 0.9855
              precision    recall  f1-score   support

           0       0.97      0.92      0.94      4202
           1       0

## Pytorch Neural Network

In [46]:
X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

X_train_processed = np.asarray(X_train_processed, dtype=np.float32)
X_test_processed = np.asarray(X_test_processed, dtype=np.float32)

In [47]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)
print(f"Number of classes: {num_classes}")

Number of classes: 3


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = TabularDataset(X_train_processed, y_train_encoded)
test_dataset = TabularDataset(X_test_processed, y_test_encoded)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

In [ ]:
import torch.nn as nn

class TabularClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [30]:
TEST_DATA_PATH = '../data/test.csv'

test_df = load_data(TEST_DATA_PATH)

In [31]:
predictions = rand_pipeline.predict(test_df)
display(pd.DataFrame({'prediction': predictions}).head(10))

,prediction
0,1
1,1
2,1
3,1
4,1
5,2
6,1
7,2
8,2
9,1


In [32]:
pred_labels = label_encoder.inverse_transform(predictions)
pred_labels

array(['Low', 'Low', 'Low', ..., 'Medium', 'Low', 'Medium'],
      shape=(270000,), dtype=object)

## Validate required ID column

In [33]:
if ID_COLUMN not in test_df.columns:
    raise ValueError(f"ID column '{ID_COLUMN}' not found in the test dataset.")

print(f"Found ID column: {ID_COLUMN}")

Found ID column: id


In [34]:
submission = pd.DataFrame({
    ID_COLUMN: test_df[ID_COLUMN],
    TARGET_COLUMN: pred_labels
})

display(submission.head())
print(submission.shape)

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


(270000, 2)


## Save submission

In [40]:
output_file = SUBMISSION_FILES / 'random_forest_submission_1.csv'
submission.to_csv(output_file, index=False)
print(f"Submission file saved to: {output_file}")

Submission file saved to: ../data/submission/random_forest_submission_1.csv
